In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print("Archive in Colab:", os.path.exists("/content/stac.tar"))
print("Project folder in Drive:",
      os.path.exists("/content/drive/MyDrive/RoofMaterialProject"))
print("Borde Rural in Drive:",
      os.path.exists("/content/drive/MyDrive/RoofMaterialProject/borde_rural"))

Archive in Colab: False
Project folder in Drive: True
Borde Rural in Drive: True


In [ ]:
import os
import shutil

folder = "/content/drive/MyDrive/RoofMaterialProject/borde_rural"

print("Borde Rural folder contents:")
for name in os.listdir(folder):
    path = os.path.join(folder, name)
    if os.path.isfile(path):
        print(f"{name} — {os.path.getsize(path) / (1024**3):.3f} GB")
    else:
        print(f"{name}/")

print("\nAvailable Colab storage:")
print(shutil.disk_usage("/content"))

Borde Rural folder contents:
borde_rural-imagery.json — 0.013 GB
borde_rural-labels-test.json — 0.000 GB
borde_rural-labels-train.json — 0.000 GB
borde_rural_ortho-cog-thumbnail.png — 0.002 GB
borde_rural_ortho-cog.tif — 4.164 GB
collection.json — 0.000 GB
test-borde_rural.geojson — 0.001 GB
train-borde_rural.geojson — 0.003 GB

Available Colab storage:
usage(total=115658190848, used=22007287808, free=93634125824)


In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  20% /


In [ ]:
import requests

url = "https://s3.amazonaws.com/drivendata-public-assets/stac.tar"

r = requests.head(url, allow_redirects=True, timeout=30)

print("HTTP status:", r.status_code)
print("File size (bytes):", r.headers.get("Content-Length"))
print("File type:", r.headers.get("Content-Type"))

HTTP status: 200
File size (bytes): 33756133376
File type: application/x-tar


In [ ]:
import requests
from tqdm.auto import tqdm

url = "https://s3.amazonaws.com/drivendata-public-assets/stac.tar"
output_path = "/content/stac.tar"

with requests.get(url, stream=True, timeout=60) as response:
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))

    with open(output_path, "wb") as f, tqdm(
        total=total, unit="B", unit_scale=True, desc="Downloading"
    ) as progress:
        for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                f.write(chunk)
                progress.update(len(chunk))

print("Download completed:", output_path)

Downloading:   0%|          | 0.00/33.8G [00:00<?, ?B/s]

Download completed: /content/stac.tar


In [ ]:
import os
import tarfile

archive_path = "/content/stac.tar"

print("Archive exists:", os.path.exists(archive_path))

if os.path.exists(archive_path):
    print(f"Archive size: {os.path.getsize(archive_path) / (1024**3):.2f} GiB")
    print("Archive readable:", tarfile.is_tarfile(archive_path))

Archive exists: True
Archive size: 31.44 GiB
Archive readable: True


In [ ]:

import tarfile

archive_path = "/content/stac.tar"

with tarfile.open(archive_path, "r:*") as tar:
    matches = [
        (member.name, member.size / (1024**3))
        for member in tar.getmembers()
        if "soach" in member.name.lower()
    ]

if matches:
    print(f"Found {len(matches)} matching entries:\n")
    for name, size_gb in matches:
        print(f"{size_gb:8.3f} GiB  {name}")
else:
    print("No entries containing 'soach' found.")


Found 9 matching entries:

   0.000 GiB  stac/colombia/borde_soacha
   0.015 GiB  stac/colombia/borde_soacha/borde_soacha-imagery.json
   0.000 GiB  stac/colombia/borde_soacha/borde_soacha-labels-test.json
   0.000 GiB  stac/colombia/borde_soacha/borde_soacha-labels-train.json
   0.001 GiB  stac/colombia/borde_soacha/borde_soacha_ortho-cog-thumbnail.png
   4.236 GiB  stac/colombia/borde_soacha/borde_soacha_ortho-cog.tif
   0.000 GiB  stac/colombia/borde_soacha/collection.json
   0.002 GiB  stac/colombia/borde_soacha/test-borde_soacha.geojson
   0.004 GiB  stac/colombia/borde_soacha/train-borde_soacha.geojson


In [ ]:

import os

drive_root = "/content/drive/MyDrive/RoofMaterialProject"

assert os.path.isdir(drive_root), (
    "Google Drive is not mounted or the project folder was not found."
)

print("Drive project folder found:", drive_root)
print("Borde Rural preserved:",
      os.path.isdir(os.path.join(drive_root, "borde_rural")))


Drive project folder found: /content/drive/MyDrive/RoofMaterialProject
Borde Rural preserved: True


In [ ]:

import os
import tarfile

archive_path = "/content/stac.tar"
output_dir = "/content/drive/MyDrive/RoofMaterialProject/borde_soacha"
prefix = "stac/colombia/borde_soacha/"

os.makedirs(output_dir, exist_ok=True)

with tarfile.open(archive_path, "r:*") as tar:
    members = [
        m for m in tar.getmembers()
        if m.name.startswith(prefix)
        and m.isfile()
    ]

    if not members:
        raise RuntimeError("No Borde Soacha files found.")

    print(f"Extracting {len(members)} files to Drive...")

    for member in members:
        relative_path = member.name[len(prefix):]
        destination = os.path.join(output_dir, relative_path)

        # Prevent extraction outside the target directory.
        if os.path.commonpath(
            [os.path.abspath(output_dir), os.path.abspath(destination)]
        ) != os.path.abspath(output_dir):
            raise RuntimeError(f"Unsafe path: {member.name}")

        os.makedirs(os.path.dirname(destination), exist_ok=True)

        # Skip an existing file if its size matches.
        if os.path.exists(destination) and os.path.getsize(destination) == member.size:
            print("Already present, skipping:", relative_path)
            continue

        source = tar.extractfile(member)
        if source is None:
            raise RuntimeError(f"Could not read: {member.name}")

        with source, open(destination, "wb") as target:
            while True:
                chunk = source.read(8 * 1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)

        print("Saved:", relative_path)

print("\nExtraction finished.")
print("Destination:", output_dir)


Extracting 8 files to Drive...
Saved: borde_soacha-imagery.json
Saved: borde_soacha-labels-test.json
Saved: borde_soacha-labels-train.json
Saved: borde_soacha_ortho-cog-thumbnail.png
Saved: borde_soacha_ortho-cog.tif
Saved: collection.json
Saved: test-borde_soacha.geojson
Saved: train-borde_soacha.geojson

Extraction finished.
Destination: /content/drive/MyDrive/RoofMaterialProject/borde_soacha
